# King County Housing — Exploratory Data Analysis

## Client

**Charles Christensen**, a seller. He wants big returns and is asking three
questions: should he renovate, which neighborhood, and when to sell.

## Data

21,597 sales of 21,420 houses in King County, WA, sold between May 2014 and
May 2015.

The data was pulled from the `eda` schema of the course database and joined
on the fly:

```sql
SELECT d.*, s.date, s.price
FROM eda.king_county_house_details AS d
INNER JOIN eda.king_county_house_sales AS s
        ON d.id = s.house_id
ORDER BY d.id, s.date;
```

The result was exported to `data/king_county_joined.csv`, which is not
tracked by git. Re-run the query above to reproduce it.

## Assumptions

- **Neighborhood** is operationalised as `zipcode`, the only geographic unit
  available in the data.
- **Return** is measured as price per square foot of living space, not
  absolute price, so that house size does not drive the result.
- **Repeat sales are real.** 176 houses were sold more than once. The
  shortest gap between two sales is 61 days and no sale date is duplicated,
  so these are genuine resales, not data-entry errors. All are kept.
- **Timing means seasonality.** The data spans a single year, so any timing
  effect is a within-year seasonal pattern, not a multi-year trend.

In [1]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {"figure.figsize": (8, 5), "axes.facecolor": "white", "axes.edgecolor": "black"}
)
plt.rcParams["figure.facecolor"] = "w"
pd.plotting.register_matplotlib_converters()
pd.set_option("display.float_format", lambda x: "%.3f" % x)

In [2]:
df = pd.read_csv("data/king_county_joined.csv")
df.shape

(21597, 21)

In [3]:
df.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price
0,1000102,6.000,3.000,2400.000,9373.000,2.000,NaN,0.000,3,7,...,0.000,1991,0.000,98002,47.326,-122.214,2060.000,7316.000,2014-09-16,280000.000
1,1000102,6.000,3.000,2400.000,9373.000,2.000,NaN,0.000,3,7,...,0.000,1991,0.000,98002,47.326,-122.214,2060.000,7316.000,2015-04-22,300000.000
2,1200019,4.000,1.750,2060.000,26036.000,1.000,NaN,0.000,4,8,...,900.000,1947,0.000,98166,47.444,-122.351,2590.000,21891.000,2014-05-08,647500.000
3,1200021,3.000,1.000,1460.000,43000.000,1.000,0.000,0.000,3,7,...,0.000,1952,0.000,98166,47.443,-122.347,2250.000,20023.000,2014-08-11,400000.000
4,2800031,3.000,1.000,1430.000,7599.000,1.500,0.000,0.000,4,6,...,420.000,1930,0.000,98168,47.478,-122.265,1290.000,10320.000,2015-04-01,235000.000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21597 non-null  int64  
 1   bedrooms       21597 non-null  float64
 2   bathrooms      21597 non-null  float64
 3   sqft_living    21597 non-null  float64
 4   sqft_lot       21597 non-null  float64
 5   floors         21597 non-null  float64
 6   waterfront     19206 non-null  float64
 7   view           21534 non-null  float64
 8   condition      21597 non-null  int64  
 9   grade          21597 non-null  int64  
 10  sqft_above     21597 non-null  float64
 11  sqft_basement  21145 non-null  float64
 12  yr_built       21597 non-null  int64  
 13  yr_renovated   17749 non-null  float64
 14  zipcode        21597 non-null  int64  
 15  lat            21597 non-null  float64
 16  long           21597 non-null  float64
 17  sqft_living15  21597 non-null  float64
 18  sqft_lot15     21

In [5]:
df.describe()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
count,21597.000,21597.000,21597.000,21597.000,21597.000,21597.000,19206.000,21534.000,21597.000,21597.000,21597.000,21145.000,21597.000,17749.000,21597.000,21597.000,21597.000,21597.000,21597.000,21597.000
mean,4580474287.771,3.373,2.116,2080.322,15099.409,1.494,0.008,0.234,3.410,7.658,1788.597,291.857,1971.000,836.651,98077.952,47.560,-122.214,1986.620,12758.284,540296.574
std,2876735715.748,0.926,0.769,918.106,41412.637,0.540,0.087,0.766,0.651,1.173,827.760,442.491,29.375,4000.111,53.513,0.139,0.141,685.230,27274.442,367368.140
min,1000102.000,1.000,0.500,370.000,520.000,1.000,0.000,0.000,1.000,3.000,370.000,0.000,1900.000,0.000,98001.000,47.156,-122.519,399.000,651.000,78000.000
25%,2123049175.000,3.000,1.750,1430.000,5040.000,1.000,0.000,0.000,3.000,7.000,1190.000,0.000,1951.000,0.000,98033.000,47.471,-122.328,1490.000,5100.000,322000.000
50%,3904930410.000,3.000,2.250,1910.000,7618.000,1.500,0.000,0.000,3.000,7.000,1560.000,0.000,1975.000,0.000,98065.000,47.572,-122.231,1840.000,7620.000,450000.000
75%,7308900490.000,4.000,2.500,2550.000,10685.000,2.000,0.000,0.000,4.000,8.000,2210.000,560.000,1997.000,0.000,98118.000,47.678,-122.125,2360.000,10083.000,645000.000
max,9900000190.000,33.000,8.000,13540.000,1651359.000,3.500,1.000,4.000,5.000,13.000,9410.000,4820.000,2015.000,20150.000,98199.000,47.778,-121.315,6210.000,871200.000,7700000.000


### First look — observations

**Shape.** 21,597 rows × 21 columns.

**Missing values.** Four columns are incomplete:

| Column | Non-null | Missing |
| --- | --- | --- |
| `waterfront` | 19,206 | 2,391 |
| `view` | 21,534 | 63 |
| `sqft_basement` | 21,145 | 452 |
| `yr_renovated` | 17,749 | 3,848 |

Every other column is complete.

**`date` is a string, not a date.** It has to be converted before any month or
season can be extracted.

**A 33-bedroom house.** `bedrooms` has a mean of 3.37 and a 75th percentile of
4, but a maximum of 33. Either a genuine mansion or a typo for 3 — needs to be
checked against its `sqft_living`.

**Heavily right-skewed columns.** `sqft_lot` has a median of 7,618 and a
maximum of 1,651,359, roughly 200× the typical lot. `price` shows the same
pattern: mean 540,297 against a median of 450,000. Wherever the mean sits well
above the median, a few very large values are stretching the right tail, so
this analysis reports **medians**, not means.

**`yr_renovated` mixes two things.** Its mean is 836.65, which is not a year at
all — it comes from averaging a column that is mostly 0 with a minority of
four-digit years. The 75th percentile is 0, so more than three quarters of the
houses were never renovated. The column is really a flag plus a date, and will
be split into both.

**Waterfront is rare.** The mean of `waterfront` is 0.008, i.e. under 1% of
sales, roughly 150 houses. Too small a group to carry the neighborhood
hypothesis on its own, so `zipcode` and coordinates will do most of that work.